# Cooling and Heating Workflows

This example illustrates the cooling and heating workflows currently employed by direct air capture technologies, which may also be utilised by other future RESKit implementations.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import reskit as rk

This script illustrates how to run a simulation with ETHOS.RESKit.CoolingHeating.


In [ ]:
# Create Placements DataFrame with turbine locations and specifications

placements = pd.DataFrame(
    {
        "lon": [5.5, 5.994685, 6.8],
        "lat": [50.797254, 50.794208, 49.5],
        "capacity": [4000, 4000, 4000],
    }
)
placements

# Run the simulation workflow for air cooling
RESKit will run the simulation and create an xarray Dataset with the simulation results for you.
Apart from the capacity_factor, RESKit also includes the conversion factors and intermediate data used to determine the capacity factor and conversion factors.

In [ ]:
reskit_xr = rk.cooling_heating.air_cooling_wenzel2025(
    placements=placements,
    era5_path=rk.TEST_DATA["era5-like"],
    temperature_coolant=30,
    design_temperature=5,
)
reskit_xr

Have a look at the previaling temperature:

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=1)
reskit_xr["surface_air_temperature"].isel(time=slice(0, 400)).plot.line(x="time", ax=ax)
ax.axhline(5, c="r")  # design Temperature, at which the system will be able to provide the specified cooling load!

RESKit will output the capacity factor of each location for every hour of the simulated year.
If the air temperature is above the design temperature, the capacity factor is lower than 1 since the designed pumps/fans would not be able to provide sufficient flows.

In [ ]:
reskit_xr["capacity_factor"].isel(time=slice(0, 400)).plot.line(x="time")

As well as the conversion factors:

In [ ]:
reskit_xr["conversion_factor_electricity"].isel(time=slice(0, 400)).plot.line(x="time")

In [ ]:
reskit_xr["electricity_input"].isel(time=slice(0, 400)).plot.line(x="time")

# Run the simulation workflow for an evaporative cooling system to calculate the water losses

In [ ]:
reskit_xr = rk.cooling_heating.evaporative_cooling_wortmann2025(
    placements=placements,
    era5_path=rk.TEST_DATA["era5-like"],
    temperatureCoolant=80,
    heatTransferDelta=10,
    efficiencyCoolingTower=0.65,
)


reskit_xr

In [ ]:
reskit_xr["conversion_factor_water"].std(dim="time")

In [ ]:
reskit_xr["conversion_factor_water"].isel(time=slice(0, 400)).plot.line(x="time")

The following plot show the influence of temperature vs. the conversion factor (specific water demand) for location 0:

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(nrows=1, ncols=1)
ax.scatter(reskit_xr["surface_air_temperature"], -reskit_xr["conversion_factor_water"])
ax.set_title("Ambient Air Temperature vs. Water Consumption")
ax.set_ylabel("Specific Water Loss [kg$_{H2O}$/kWh$_{th}$]")
ax.set_xlabel("Air Temperature [°C]")

# Run the simulation workflow for air source heat pumps
RESKit will run the simulation and create an xarray Dataset with the simulation results for you.

In [ ]:
reskit_hp = rk.cooling_heating.air_source_heat_pump(placements=placements, era5_path=rk.TEST_DATA["era5-like"])
reskit_hp

In [ ]:
reskit_hp["COP"].isel(time=slice(0, 400)).plot.line(x="time")